In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

events = pd.read_parquet(
    PROCESSED_DIR / "nse_reversal_model_dataset_v1.parquet"
)

events["Date"] = pd.to_datetime(events["Date"])
events = events.sort_values(["Date", "ticker"]).reset_index(drop=True)

TRAIN_END = pd.Timestamp("2022-12-31")
VALID_START = pd.Timestamp("2023-01-01")
VALID_END = pd.Timestamp("2024-12-31")
TEST_START = pd.Timestamp("2025-01-01")

print("Date range:", events["Date"].min(), "to", events["Date"].max())
print("Total rows:", len(events))

Date range: 2015-02-19 00:00:00 to 2026-06-12 00:00:00
Total rows: 14949


In [2]:
train_raw = events[events["Date"] <= TRAIN_END].copy()

valid_raw = events[
    (events["Date"] >= VALID_START)
    & (events["Date"] <= VALID_END)
].copy()

test_raw = events[events["Date"] >= TEST_START].copy()

print("Raw split sizes:")
print("Train:", len(train_raw))
print("Validation:", len(valid_raw))
print("Test:", len(test_raw))

Raw split sizes:
Train: 11313
Validation: 1851
Test: 1785


In [3]:
PURGE_DAYS = 5

# Unique trading dates present in the event dataset
all_dates = np.array(sorted(events["Date"].unique()))

def purge_last_trading_dates(split_df, n_dates=5):
    split_dates = np.array(sorted(split_df["Date"].unique()))
    dates_to_remove = split_dates[-n_dates:]

    return split_df[
        ~split_df["Date"].isin(dates_to_remove)
    ].copy(), dates_to_remove

train, train_purged_dates = purge_last_trading_dates(
    train_raw,
    PURGE_DAYS
)

valid, valid_purged_dates = purge_last_trading_dates(
    valid_raw,
    PURGE_DAYS
)

# Test is never used to train or tune, so we keep all of it.
test = test_raw.copy()

print("Purged train dates:", train_purged_dates)
print("Purged validation dates:", valid_purged_dates)

print("\nFinal split sizes:")
print("Train:", len(train))
print("Validation:", len(valid))
print("Test:", len(test))

Purged train dates: [Timestamp('2022-12-23 00:00:00') Timestamp('2022-12-26 00:00:00')
 Timestamp('2022-12-27 00:00:00') Timestamp('2022-12-28 00:00:00')
 Timestamp('2022-12-29 00:00:00')]
Purged validation dates: [Timestamp('2024-12-24 00:00:00') Timestamp('2024-12-26 00:00:00')
 Timestamp('2024-12-27 00:00:00') Timestamp('2024-12-30 00:00:00')
 Timestamp('2024-12-31 00:00:00')]

Final split sizes:
Train: 11288
Validation: 1771
Test: 1785


In [4]:
print("Train last date:", train["Date"].max())
print("Validation first date:", valid["Date"].min())

print("\nValidation last date:", valid["Date"].max())
print("Test first date:", test["Date"].min())

assert (
    train["Date"].max()
    < valid["Date"].min()
), "Train/validation dates overlap."

assert (
    valid["Date"].max()
    < test["Date"].min()
), "Validation/test dates overlap."

print("\nChronological separation check passed.")

Train last date: 2022-12-22 00:00:00
Validation first date: 2023-01-02 00:00:00

Validation last date: 2024-12-23 00:00:00
Test first date: 2025-01-01 00:00:00

Chronological separation check passed.


In [5]:
def split_summary(name, df):
    counts = df["reversal"].value_counts().sort_index()
    positive_rate = df["reversal"].mean() * 100

    return {
        "split": name,
        "rows": len(df),
        "no_reversal": counts.get(0, 0),
        "reversal": counts.get(1, 0),
        "reversal_rate_pct": round(positive_rate, 2),
        "start_date": df["Date"].min().date(),
        "end_date": df["Date"].max().date(),
    }

split_summary_df = pd.DataFrame([
    split_summary("train", train),
    split_summary("validation", valid),
    split_summary("test", test),
])

split_summary_df

,split,rows,no_reversal,reversal,reversal_rate_pct,start_date,end_date
0,train,11288,8211,3077,27.26,2015-02-19,2022-12-22
1,validation,1771,1421,350,19.76,2023-01-02,2024-12-23
2,test,1785,1398,387,21.68,2025-01-01,2026-06-12


In [6]:
train.to_parquet(PROCESSED_DIR / "train_events_v1.parquet", index=False)
valid.to_parquet(PROCESSED_DIR / "validation_events_v1.parquet", index=False)
test.to_parquet(PROCESSED_DIR / "test_events_v1.parquet", index=False)

split_summary_df.to_csv(
    PROCESSED_DIR / "split_summary_v1.csv",
    index=False
)

print("Saved train, validation, test splits.")

Saved train, validation, test splits.
